# ASR Hutsul — Colab Training Notebook

Main entry point for running the full ASR Hutsul reproduction
project on Google Colab.  All training artefacts persist on
Google Drive at:

```
/content/drive/MyDrive/hutsul_asr/
├── checkpoints/<variant>/        ← every training checkpoint-N
├── final_models/<variant>/       ← post-training save_model output
├── preprocessed/<model_type>/    ← cached normalized DatasetDict
├── evaluations/{csv,json,predictions}/
├── tensorboard/<variant>/        ← TensorBoard event files
├── cache/                        ← HF_HOME / TRANSFORMERS_CACHE
└── datasets/                     ← HF_DATASETS_CACHE
```

The repository itself stays in **local** Colab storage at
`/content/asr_hutsul_reproduction/` so editing/pulling is fast.
Only outputs/checkpoints/cache go to Drive — to survive runtime
disconnects.

## Sections

1. Environment setup — GPU check, install deps.
2. Mount Google Drive.
3. Clone or `git pull` the repository.
4. Hugging Face token.
5. Verify storage layout + redirect HF caches.
6. Preprocess the dataset.
7. Training launchers (one cell per variant).
8. Resource-aware Colab settings.
9. Evaluation.
10. Inference demo.
11. TensorBoard.
12. Troubleshooting.

## 1. Environment setup

In [ ]:
import subprocess
import torch

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda)
print('cuDNN   :', torch.backends.cudnn.version())
print('GPU     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')
if torch.cuda.is_available():
    print()
    print(subprocess.check_output(['nvidia-smi']).decode())

In [ ]:
%pip install -q --upgrade \
    'transformers>=4.45,<4.50' \
    'datasets>=2.20,<3.3' \
    'evaluate>=0.4.2,<0.5' \
    'accelerate>=0.34,<1.5' \
    'peft>=0.12,<0.15' \
    'jiwer>=3.0.4' \
    'librosa>=0.10.2' \
    'soundfile>=0.12.1' \
    'audiomentations>=0.36.0' \
    'tensorboard>=2.16' \
    'PyYAML>=6.0.1'

If pip prints messages about restarting the runtime, do so via
**Runtime → Restart runtime** before continuing — re-run only this
section, no need to reinstall packages.

## 2. Mount Google Drive

Drive is the canonical storage root.  Everything except the
repository source lives under
`/content/drive/MyDrive/hutsul_asr/`.

In [ ]:
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/hutsul_asr')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ('checkpoints', 'final_models', 'preprocessed',
            'evaluations', 'evaluations/csv', 'evaluations/json',
            'evaluations/predictions',
            'tensorboard', 'cache', 'datasets'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print('Drive ready:', DRIVE_ROOT)
print('Listing:')
for p in sorted(DRIVE_ROOT.iterdir()):
    print(' ', p.name)

## 3. Clone or `git pull` the repository

The repo itself stays in `/content/asr_hutsul_reproduction/` — fast
local SSD.  Only outputs and caches use Drive.  The cell below
supports two workflows:

* **Active development**: fill `REPO_URL` with your fork and run a
  fresh `git clone` on first use, then `git pull` on subsequent
  reconnects.
* **Drive-bundled source**: place a snapshot under
  `<DRIVE_ROOT>/src/asr_hutsul_reproduction/` and the cell will
  copy it locally.

In [ ]:
import os
import sys
import shutil

REPO_URL = ''  # e.g. 'https://github.com/<you>/asr_hutsul_reproduction.git'
WORK_DIR = Path('/content/asr_hutsul_reproduction')

if REPO_URL:
    if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
        print('Existing clone — pulling latest...')
        !cd {WORK_DIR} && git pull --ff-only
    elif WORK_DIR.exists():
        raise SystemExit(
            f'{WORK_DIR} exists but is not a git repo — remove it manually.'
        )
    else:
        !git clone {REPO_URL} {WORK_DIR}
else:
    src = DRIVE_ROOT / 'src' / 'asr_hutsul_reproduction'
    if WORK_DIR.exists():
        print(f'Reusing existing working tree at {WORK_DIR}')
    elif src.exists():
        print(f'Copying project from {src}...')
        shutil.copytree(src, WORK_DIR)
    else:
        raise SystemExit(
            'No REPO_URL set, and no Drive snapshot at\n'
            f'{src}.\n'
            'Either set REPO_URL above or upload the project there.'
        )

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print('CWD :', os.getcwd())
print('Top-level files:')
for p in sorted(WORK_DIR.iterdir()):
    if p.name.startswith('.') and p.name != '.gitkeep':
        continue
    print(' ', p.name)

In [ ]:
# Install the project's pinned dependencies on top of the bulk
# install above.
%pip install -q -r requirements.txt

## 4. Hugging Face token

The dataset is public; some base checkpoints (Whisper-large-v3,
`Yehor/w2v-bert-uk-v2.1`) are gated and need a personal token.
Either paste below or pre-store it via **Runtime → Manage
secrets** (key `HF_TOKEN`).

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    HF_TOKEN = input('Paste HF token (leave blank to skip): ').strip()

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN configured.')
else:
    print('Continuing without an HF token (works for the public dataset).')

## 5. Verify storage layout

Force the project's `StorageLayout` helper to point at our Drive
root and propagate that to the HF caches.  Subprocesses spawned by
`!python train.py` inherit the env vars set here.

In [ ]:
os.environ['HUTSUL_ASR_ROOT'] = str(DRIVE_ROOT)

from config import resolve_storage_layout, configure_hf_caches  # noqa: E402

layout = resolve_storage_layout(DRIVE_ROOT, refresh=True)
layout.ensure()
configure_hf_caches(layout)
print(layout.summary())

# Confirm the env vars children processes will see.
for k in ('HUTSUL_ASR_ROOT', 'HF_HOME', 'TRANSFORMERS_CACHE',
          'HF_DATASETS_CACHE', 'HF_HUB_DISABLE_TELEMETRY'):
    print(f'{k:>28s} = {os.environ.get(k)}')

## 6. Dataset preprocessing

Loads the dataset, resamples to 16 kHz, normalises transcripts and
writes an 80/10/10 split to
`<DRIVE_ROOT>/preprocessed/shared/`.  The trainers point each
model family at its own `preprocessed/<model_type>/` subdirectory
but reuse the same upstream Hugging Face dataset cache.

In [ ]:
import logging
from config import ProjectConfig, configure_logging
from preprocess import load_and_prepare
configure_logging(level=logging.INFO)

project_cfg = ProjectConfig(
    preprocessed_dir=layout.preprocessed_dir('shared'),
    dataset_cache_dir=layout.datasets_cache,
    cache_dir=layout.cache,
    output_dir=layout.root,
    log_dir=layout.tensorboard,
)
project_cfg.ensure_dirs()

dataset, audio_col, text_col = load_and_prepare(
    project_cfg,
    token=HF_TOKEN or None,
)
for split, ds in dataset.items():
    print(f'{split:>10s}: {len(ds):>6} samples')
print('audio column =', audio_col)
print('text  column =', text_col)

## 7. Training launchers

One cell per model variant.  All paths are auto-resolved through
the `StorageLayout` because `HUTSUL_ASR_ROOT` is exported.  Use
`--resume_from_checkpoint LATEST` (already in every cell) to
transparently recover after Colab disconnects.

### 7.1 Whisper-small (T4 OK)

In [ ]:
!python train.py \
    --model_type whisper \
    --variant whisper-small \
    --resume_from_checkpoint LATEST

### 7.2 Whisper-medium (L4 / A100)

In [ ]:
!python train.py \
    --model_type whisper \
    --variant whisper-medium \
    --resume_from_checkpoint LATEST

### 7.3 Whisper-large-v3 (A100)

In [ ]:
!python train.py \
    --model_type whisper \
    --variant whisper-large-v3 \
    --resume_from_checkpoint LATEST

### 7.4 arampacha/whisper-large-uk-2 (A100)

In [ ]:
!python train.py \
    --model_type whisper \
    --variant whisper-large-uk-2 \
    --resume_from_checkpoint LATEST

### 7.5 Wav2Vec2-XLSR (T4 OK)

In [ ]:
!python train.py \
    --model_type wav2vec2 \
    --variant xlsr-300m-uk \
    --resume_from_checkpoint LATEST

### 7.6 Wav2Vec2-BERT-UK-v2.1 (L4 / A100)

In [ ]:
!python train.py \
    --model_type wav2vec2_bert \
    --variant w2v-bert-uk-v2.1 \
    --resume_from_checkpoint LATEST

### 7.7 OmniASR-300M (L4 / A100)

In [ ]:
!python train.py \
    --model_type omniasr \
    --variant omniasr-300m \
    --resume_from_checkpoint LATEST

### 7.8 OmniASR-1B (A100)

In [ ]:
!python train.py \
    --model_type omniasr \
    --variant omniasr-1b \
    --resume_from_checkpoint LATEST

## 8. Resource-aware Colab settings

| GPU        | Comfortable variants                              |
|------------|---------------------------------------------------|
| T4 16 GB   | whisper-small, xlsr-300m-uk (lower batch_size)    |
| L4 22 GB   | whisper-medium, w2v-bert-uk-v2.1, omniasr-300m    |
| A100 40 GB | whisper-large-v3, omniasr-1b                      |

Override knobs:

* `--batch_size`  (per-device train batch),
* `--grad_accum`  (gradient accumulation steps),
* `--max_steps`   (cap for short Colab sessions),
* `--bf16`        (A100/H100 only — slightly faster than fp16).

In [ ]:
# Example: low-VRAM Whisper-medium recipe on T4.
!python train.py \
    --model_type whisper \
    --variant whisper-medium \
    --batch_size 2 \
    --grad_accum 16 \
    --max_steps 4000 \
    --resume_from_checkpoint LATEST

## 9. Evaluation

Once training finishes, evaluate the saved final model.  Output
files land under `evaluations/{csv,json,predictions}/<run_name>/`.

In [ ]:
VARIANT = 'whisper-small'   # set to whichever you trained
FINAL = layout.final_model_dir(VARIANT)

!python evaluate.py \
    --checkpoint "{FINAL}" \
    --run_name "{VARIANT}" \
    --split test \
    --batch_size 8

In [ ]:
import json
results = json.loads((layout.evaluations_json / VARIANT / 'test_results.json').read_text())
print(json.dumps(results, indent=2, ensure_ascii=False))

## 10. Inference demo

In [ ]:
import torch
import numpy as np
import librosa
from evaluate import load_model_and_processor
from utils.text_normalization import build_default_normalizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded = load_model_and_processor(
    layout.final_model_dir(VARIANT),
    device=device,
    hf_token=HF_TOKEN or None,
)

from google.colab import files  # type: ignore
uploaded = files.upload()
AUDIO_PATH = Path(next(iter(uploaded.keys())))

samples, _ = librosa.load(str(AUDIO_PATH), sr=16_000, mono=True)
samples = samples.astype(np.float32)
batch = loaded.feature_extractor([samples], sampling_rate=16_000, return_tensors='pt', padding=True)
batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
loaded.model.eval()
with torch.no_grad():
    if loaded.family == 'whisper':
        pred_ids = loaded.model.generate(**batch, max_new_tokens=225, language='uk', task='transcribe')
    else:
        pred_ids = loaded.model(**batch).logits.argmax(dim=-1)
raw = loaded.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]
norm = build_default_normalizer()(raw)
print('Transcription:', norm)

## 11. TensorBoard

All variants share the same root and show up as separate runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {layout.tensorboard}

## 12. Troubleshooting

### CUDA out of memory
* Reduce `--batch_size`, raise `--grad_accum` to keep effective
  batch size constant.
* Drop from `whisper-large-v3` to `whisper-medium` on T4.
* Make sure `gradient_checkpointing` stays on (it is the default).

### Drive quota / permission error
* `<DRIVE_ROOT>` mostly stores small files (configs, json, csv) +
  large model checkpoints.  Watch out for the 15-GB free quota.
* Re-mount Drive (`drive.mount('/content/drive', force_remount=True)`).

### HF token issues
* Re-run section 4.  Free Colab loses environment variables on
  every reconnect; the **Manage secrets** drawer persists them.

### Deprecated argument errors
* If a Trainer raises `TypeError: ... 'evaluation_strategy'` you
  have an old transformers install.  Re-run section 1.

### Colab disconnect recovery
* Reconnect, re-run sections 1–5, then re-launch the same
  training cell.  `--resume_from_checkpoint LATEST` already in
  every cell will pick up where you left off — checkpoints are on
  Drive, so they survive the disconnect.

### Resume jumps back to step 0
* Make sure `HUTSUL_ASR_ROOT` is set (section 5) and `--output_dir`
  is *not* manually overridden.  The auto-resolution must point at
  the same `<DRIVE_ROOT>/checkpoints/<variant>/` you trained into.
* Check `<DRIVE_ROOT>/checkpoints/<variant>/checkpoint-*/` for
  empty/zero-byte directories from a killed save and remove them.